# 펭귄 데이터셋 직접 EDA 해보기

**펭귄의 몸 크기와 모양은 종에 따라 어떻게 다를까요?**

Iris에서 살펴본 방법을 펭귄 데이터에 적용합니다. 결측치 처리는 강사와 함께 하고, 이후에는 Continue에 코드를 요청해 실행합니다.

**진행 순서:** 준비 → 결측치 함께 처리 → 기초 통계와 분포 → 상관관계와 산점도 → 자유 탐색

- **함께 진행:** Notion의 요청 예시로 코드를 받아 실행하고 결과를 살펴봅니다.
- **직접 탐색:** 결과에서 궁금한 내용을 골라 자신의 말로 요청합니다.

코드 문법을 외울 필요는 없습니다. 받은 코드를 해당 단계의 빈 코드 셀에 붙여넣어 실행하세요.

## 1. 어떤 데이터인가요?

펭귄의 신체 측정값과 종, 성별, 관찰된 섬을 담은 데이터입니다. 이번에는 측정값의 분포와 종별 차이를 탐색합니다.

**한 행은 펭귄 한 마리를 관찰한 기록입니다.**

| 컬럼 | 의미 | 단위 |
|---|---|---|
| species | 펭귄 종: Adelie, Chinstrap, Gentoo | 범주 |
| island | 관찰된 섬: Biscoe, Dream, Torgersen | 범주 |
| bill_length_mm | 부리 길이 | mm |
| bill_depth_mm | 부리 두께(위아래 두께) | mm |
| flipper_length_mm | 날개(지느러미) 길이 | mm |
| body_mass_g | 체중 | g |
| sex | 성별: Male, Female | 범주 |

species는 우리가 구분하려는 대상이고, 나머지 항목은 신체 측정값과 관찰 정보입니다.

![penguin](penguin.png)

## 2. 실습 준비

아래 준비 셀을 먼저 실행하세요. 표는 `df`라는 이름으로 불러오며, 한글 그래프 설정도 함께 준비합니다.

seaborn에서 제공하는 펭귄 데이터를 불러옵니다. 처음 불러올 때는 인터넷 연결이 필요합니다.

데이터 출처: [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) · [수업에서 사용하는 seaborn CSV](https://github.com/mwaskom/seaborn-data/blob/master/penguins.csv)

In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display

sns.set_style("whitegrid")
available_fonts = {font.name for font in fm.fontManager.ttflist}
korean_font = next((name for name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]
                    if name in available_fonts), None)
if korean_font:
    plt.rcParams["font.family"] = korean_font
else:
    print("한글 글꼴을 찾지 못했습니다.")
plt.rcParams["axes.unicode_minus"] = False

df_raw = sns.load_dataset("penguins")
df = df_raw.copy()
display(df.head())

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


## 3. 결측치 함께 처리하기

결측치는 측정되지 않았거나 기록되지 않은 값이며, 표에서는 `NaN`으로 보입니다. 우선 데이터의 정보를 출력해 봅시다.

In [12]:
print(f"전체: {len(df_raw)}행, {len(df_raw.columns)}열")
display(df_raw.info())
display(df_raw.isna().sum().rename("결측치 수"))

전체: 344행, 7열
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


None

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
Name: 결측치 수, dtype: int64

확인해 보면 `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex` 5개 컬럼에서 결측치가 발견되었습니다.

결측치는 처리하는 방법이 정해져 있는 것이 아니라 데이터를 살펴보고 결정하는 것입니다. 우선 결측 데이터를 확인해 보도록 하겠습니다.

In [13]:
display(df_raw[df_raw.isna().any(axis=1)])

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,NaN
11,Adelie,Torgersen,37.8,17.3,180.0,3700.0,NaN
47,Adelie,Dream,37.5,18.9,179.0,2975.0,NaN
246,Gentoo,Biscoe,44.5,14.3,216.0,4100.0,NaN
286,Gentoo,Biscoe,46.2,14.4,214.0,4650.0,NaN
324,Gentoo,Biscoe,47.3,13.8,216.0,4725.0,NaN
336,Gentoo,Biscoe,44.5,15.7,217.0,4875.0,NaN


### 처리 방법과 이번 실습의 선택

- **제외하기:** 분석에 필요한 값이 없는 행을 제외합니다. 사용할 자료가 줄어듭니다.
- **채우기:** 평균·중앙값 등으로 채울 수 있지만, 분포나 관계가 달라질 수 있습니다.

이 데이터는 344행 중 같은 2행에 신체 측정값 네 가지가 모두 빠져 있습니다. 이번에는 신체 측정값을 탐색하므로 **이 2행만 제외**합니다. 성별만 빠진 기록은 남겨두고, 원본은 `df_raw`에 보관합니다.

In [14]:
measurements = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df = df_raw.dropna(subset=measurements).copy()
print(f"처리 전 {len(df_raw)}행 → 처리 후 {len(df)}행 (제외: {len(df_raw) - len(df)}행)")
display(df.isna().sum().rename("남은 결측치 수"))

처리 전 344행 → 처리 후 342행 (제외: 2행)


species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  9
Name: 남은 결측치 수, dtype: int64

신체 측정값이 있는 **342행**으로 이후 분석을 진행합니다. 성별 결측치는 9개 남아 있으며, 성별로 비교할 때만 해당 기록을 제외하면 됩니다.

## 4. Continue로 분석 시작하기

[Notion 요청 예시](https://app.notion.com/p/EDA_practice-ipynb-3dab39501c38807bbb29dc461b9b645d)를 열어 **처음 한 번 전달할 지침**을 Continue에 입력하세요. 노트북의 데이터 설명과 준비 코드를 대화에 함께 제공하면 맥락을 전달하기 쉽습니다.

이후 같은 대화에서 단계별로 요청합니다. 오류가 나면 오류 메시지와 실행한 코드를 함께 전달하세요. 결과 해석을 물을 때도 실제 표나 그래프를 함께 전달합니다.

## 5. 기초 통계와 분포 살펴보기

### 함께 진행 · 기초 통계

Notion의 **5. 기초 통계** 요청 예시를 사용하세요. 네 가지 신체 측정값의 평균·중앙값·최솟값·최댓값을 확인합니다. 평균은 전체 값을 더해 개수로 나눈 값이고, 중앙값은 크기순으로 놓았을 때 가운데 값입니다. 항목마다 단위가 다르므로 함께 확인하세요.

### 직접 탐색 · 숫자를 분포로 확인하기

통계표에서 관심 있는 항목 하나를 골라 **히스토그램을 그리는 코드**를 자신의 말로 요청하세요. 가로축은 선택한 측정값, 세로축은 각 구간에 속하는 펭귄 수입니다.

값이 어디에 몰려 있고 얼마나 넓게 퍼져 있는지 살펴보세요.

종별 차이가 궁금하다면 같은 항목의 분포를 종별로 나누어 확인해도 좋습니다. 한 번 종 별로 나눠서 히스토그램을 그려서 비교해 보세요.

## 6. 상관관계와 산점도 살펴보기

### 함께 진행 · 상관관계 히트맵

Notion의 **6. 상관관계 히트맵** 요청 예시를 사용하세요. 각 칸은 두 측정값의 상관계수이며, 색과 숫자로 관계의 방향과 강도를 보여줍니다.

+1에 가까우면 함께 커지는 경향, −1에 가까우면 한쪽이 커질수록 다른 쪽이 작아지는 경향입니다. 0에 가까우면 직선적인 관계가 약합니다. 자기 자신과 비교하는 대각선은 1이므로 탐색에서 제외합니다.

### 직접 탐색 · 두 항목을 골라 산점도로 확인하기

대각선을 제외하고 **상관계수의 절댓값이 큰 두 항목**을 골라 산점도를 요청하세요. 음수도 −1에 가까우면 강한 관계입니다. 점 하나는 펭귄 한 마리이고, 두 축은 선택한 측정값입니다.

점들이 모이는 방향과 흩어진 정도를 확인한 뒤, **종별로 점의 색을 나누도록 추가 요청**하세요. 전체에서 보인 경향이 같은 종 안에서도 나타나는지 살펴봅니다. 필요한 만큼 코드 셀을 추가해도 됩니다.

## 7. 궁금한 내용 더 확인하기

아래 세 가지 중 **하나를 골라**, 확인할 내용을 자신의 말로 Continue에 요청하세요. 받은 코드를 실행하고 표나 그래프로 결과를 확인합니다.

### 선택 1. 같은 종에서도 성별에 따라 체중이 다를까?

종(species)별로 암컷·수컷(sex)의 체중(body_mass_g)을 비교하세요.

- **확인할 내용:** 종과 성별로 나눈 평균·중앙값·마릿수, 체중 분포
- **분석 방법:** 그룹별 요약표와 상자그림. 상자그림은 가운데 선으로 중앙값을, 상자로 가운데 50%의 값이 모인 범위를 보여줍니다.
- 성별이 없는 기록은 이 분석에서만 제외합니다. 평균 차이와 함께 분포가 얼마나 겹치는지도 살펴보세요.

### 선택 2. 전체에서 보인 상관관계가 종별로도 비슷할까?

앞의 산점도에서 선택한 두 측정값으로 이어서 분석하세요.

- **확인할 내용:** 전체 상관계수와 각 종 안에서 계산한 상관계수의 차이
- **분석 방법:** 전체·종별 상관계수와 분석에 사용한 마릿수를 한 표로 비교
- 관계의 방향과 강도가 비슷한지 확인하고, 앞에서 그린 종별 색상 산점도와 함께 해석하세요.

### 선택 3. 섬마다 펭귄 종의 구성이 다를까?

섬(island)별로 어떤 종(species)이 몇 마리씩 기록되어 있는지 확인하세요.

- **확인할 내용:** 섬별 종의 마릿수와 각 섬 안에서 종별로 차지하는 비율
- **분석 방법:** 섬×종 개수표와 섬별 종 비율 막대그래프
- 섬마다 전체 기록 수가 다르므로 개수와 비율을 함께 비교하세요. 여기서의 비율은 **이 데이터에 기록된 펭귄의 구성**이며, 섬 전체의 실제 개체 수 비율을 뜻하지 않습니다.

AI에게 해석을 물을 때는 실행 결과를 함께 전달하고, 설명이 실제 표·그래프와 맞는지 확인하세요. 상관관계가 원인을 뜻하거나, 종별로 점이 나뉘어 보인다고 예측 성능이 검증된 것은 아닙니다.